# Funciones Lambda y la Potencia de `.apply()`

## 🎯 Objetivos
En el análisis de datos, a menudo necesitamos realizar transformaciones rápidas y sencillas que no justifican la creación de una función completa con `def`. Para ello, Python nos ofrece las **funciones lambda**. En este notebook aprenderás a:
1. Comprender la naturaleza y sintaxis de las funciones lambda (funciones anónimas).
2. Integrar funciones lambda dentro del método `.apply()` para transformaciones eficientes.
3. Diferenciar entre el uso de lambdas y las funciones estándar `def`.
4. Identificar cuándo utilizar los accesorios de pandas (`.str`, `.dt`) en lugar de una lambda para mejorar el rendimiento.

## 💡 Introducción

Una función **lambda** es una pequeña función anónima (sin nombre) que se define en una sola línea. Se llaman "anónimas" porque no necesitan un nombre para existir; se crean, se usan y luego desaparecen.

Son ideales para ser pasadas como argumentos a otras funciones, como `.apply()`, permitiéndonos escribir transformaciones concisas y elegantes sin llenar nuestro código de funciones pequeñas que solo usaremos una vez.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Configuración del dataset
file_path = Path('players_20.csv')
df = pd.read_csv(file_path)
df.set_index('short_name', inplace=True)
df = df[['long_name', 'age', 'dob', 'height_cm', 'weight_kg', 'nationality', 'club']]

df.head()

## 🛠️ Entendiendo la Función Lambda

La sintaxis básica es: `lambda argumentos: expresión`

### 🌉 Puente Pedagógico: Función `def` vs `lambda`
Imagina que una función `def` es como construir una **cocina completa** en tu casa: tiene nombre, espacio definido y herramientas permanentes. Es ideal para recetas complejas que repetirás muchas veces.

Una función `lambda` es como un **food truck**: es compacto, se despliega rápidamente, cumple una función específica y se mueve al siguiente lugar. Es ideal para tareas rápidas y sencillas.

```
   FUNCION def (Edificio)      --->      FUNCION lambda (Kiosco)
   +---------------------+             +----------------------+
   | def sumar(a, b):     |             | lambda a, b: a + b   |
   |     return a + b     |             +----------------------+
   +---------------------+             (Sin nombre, una sola línea)
```

In [ ]:
# Definición tradicional
def sumar_tradicional(a, b):
    return a + b

# Definición lambda
sumar_lambda = lambda a, b: a + b

print(f"Resultado tradicional: {sumar_tradicional(2, 3)}")
print(f"Resultado lambda: {sumar_lambda(2, 3)}")

## 🚀 Combinando `.apply()` y `lambda`

La verdadera magia ocurre cuando pasamos una lambda directamente a `.apply()`. Esto elimina la necesidad de asignar la función a una variable.

### 1. Transformaciones Numéricas
Convertir los centímetros de la columna `height_cm` a metros.

In [ ]:
# Aplicamos la lambda directamente
df['height_m'] = df['height_cm'].apply(lambda x: x / 100)

df[['height_cm', 'height_m']].head()

### 2. Transformaciones de Texto
Convertir los nombres largos a mayúsculas.

In [ ]:
# Usando lambda
nombres_mayus = df['long_name'].apply(lambda x: x.upper())

print("Usando lambda:")
print(nombres_mayus.head(3))

# 💡 EL CAMINO PANDAS: Usar .str es más eficiente y legible
nombres_mayus_opt = df['long_name'].str.upper()
print("\nUsando .str.upper() (Optimizado):")
print(nombres_mayus_opt.head(3))

### 3. Transformaciones de Fecha
Extraer el año de nacimiento de la columna `dob`.

In [ ]:
# Primero aseguramos que la columna sea tipo datetime
df['dob'] = pd.to_datetime(df['dob'])

# Usando lambda
anios_nac = df['dob'].apply(lambda x: x.year)

print("Usando lambda:")
print(anios_nac.head(3))

# 💡 EL CAMINO PANDAS: Usar .dt es la forma correcta
anios_nac_opt = df['dob'].dt.year
print("\nUsando .dt.year (Optimizado):")
print(anios_nac_opt.head(3))

### 4. Lambdas en DataFrames (Múltiples Columnas)

Cuando usamos `axis=1`, el argumento `x` de la lambda ya no es un valor, sino una **fila completa**.

In [ ]:
# Calcular el IMC en una sola línea usando lambda y axis=1
df['IMC'] = df.apply(lambda row: row['weight_kg'] / ((row['height_cm'] / 100) ** 2), axis=1)

df[['long_name', 'IMC']].head()

## 📝 Ejercicios de Práctica

1. **Cálculo Simple**: Crea una lambda que multiplique la edad de los jugadores por 2 y aplícala a la columna `age`.
2. **Slicing de Texto**: Utiliza una lambda para extraer solo los primeros 3 caracteres de la columna `nationality`.
3. **Lógica Condicional**: Crea una lambda que devuelva "Pesado" si el `weight_kg` es mayor a 85 y "Ligero" en caso contrario. Aplícala al DataFrame para crear la columna `peso_cat`.

## 📋 Resumen Rápido

| Característica | Función `def` | Función `lambda` |
| :--- | :--- | :--- | :--- |
| **Nombre** | Obligatorio | Anónima |
| **Sintaxis** | Múltiples líneas | Una sola línea |
| **Reutilización** | Alta (definida una vez, usada muchas) | Baja (usualmente desechable) |
| **Uso ideal** | Lógica compleja y repetitiva | Transformaciones rápidas dentro de `.apply()` |